# Base Messages

The `base.py` module defines the shared message structure used by LangChain chat models. It provides base message classes, streaming message chunks, text extraction, content-block conversion, serialization helpers, and content-merging utilities.

# BaseMessage: `Serializable`
`BaseMessage` represents the common message structure used by LangChain chat models. It stores message content, provider-specific data, response metadata, message type, an optional name, and an optional identifier.

## Fields
1. `content`:`str | list[str | dict[Any, Any]]`:= Stores the contents of the message as plain text or as a list containing strings and dictionary-based content blocks.
2. `additional_kwargs`:`dict[Any, Any]`:= Stores additional provider-specific payload data associated with the message. Its default value is an empty dictionary.
3. `response_metadata`:`dict[Any, Any]`:= Stores response information such as headers, log probabilities, token counts, and model names. Its default value is an empty dictionary.
4. `type`:`str`:= Stores the unique message type used during serialization and deserialization.
5. `name`:`str | None`:= Stores an optional human-readable name for the message. Its default value is `None`.
6. `id`:`str | None`:= Stores an optional unique identifier for the message. Numeric identifiers are automatically converted to strings.

## Configuration
1. `model_config`:`ConfigDict`:= Allows additional fields that are not explicitly declared in the model.

   ```python
   model_config = ConfigDict(
       extra="allow"  # Allow undeclared fields
   )
   ```

## Properties
1. `content_blocks`:`list[types.ContentBlock]`:= Converts the message content into standardized LangChain content blocks.

   Plain strings are converted into text blocks. Recognized blocks are preserved, while provider-specific or older formats are converted when possible. Unsupported formats are retained as non-standard blocks.

2. `text`:`TextAccessor`:= Returns only the textual content of the message.
   For list-based content, only plain strings and dictionaries with `type="text"` are included. Other content-block types are ignored.

## Methods
1. `__init__`:= Initializes a message using raw content or standardized content blocks.
   ```python
   __init__(
       self,
       content: str | list[str | dict[Any, Any]] | None = None,  # Raw message content
       content_blocks: list[types.ContentBlock] | None = None,  # Standardized content blocks
       **kwargs: Any  # Additional message fields
   ) -> None
   ```

2. `is_lc_serializable`:= Indicates whether `BaseMessage` supports LangChain serialization.
   ```python
   @classmethod
   is_lc_serializable(
       cls  # BaseMessage class
   ) -> bool
   ```

3. `get_lc_namespace`:= Returns the LangChain serialization namespace used for messages.
   ```python
   @classmethod
   get_lc_namespace(
       cls  # BaseMessage class
   ) -> list[str]
   ```

4. `__add__`:= Combines the current message with another message-like object and returns a `ChatPromptTemplate`.
   ```python
   __add__(
       self,
       other: Any  # Message or message-like object to concatenate
   ) -> ChatPromptTemplate
   ```

5. `pretty_repr`:= Returns a readable representation of the message.
   The representation includes a formatted title, the optional message name, and the message content.

   ```python
   pretty_repr(
       self,
       html: bool = False  # Whether to use HTML-style formatting
   ) -> str
   ```

6. `pretty_print`:= Prints a readable representation of the message.
   ```python
   pretty_print(self) -> None
   ```

In [1]:
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage


message = HumanMessage(
    content="Explain SQL joins.", # Message content
    name="student", # Optional message name
    id=101, # Numeric ID is converted to string
    additional_kwargs={
        "priority": "high" # Provider-specific data
    },
    response_metadata={
        "model_name": "example-model", # Model information
        "token_count": 10 # Token usage information
    },
    custom_field="custom value" # Allowed because extra="allow"
)


print("Content:", message.content) # Raw message content
print("Additional kwargs:", message.additional_kwargs) # Provider-specific data
print("Response metadata:", message.response_metadata) # Response-related metadata
print("Type:", message.type) # Message type
print("Name:", message.name) # Optional message name
print("ID:", message.id) # Identifier stored as string
print("Custom field:", message.custom_field) # Additional undeclared field

print("Content blocks:", message.content_blocks) # Standardized content blocks
print("Text:", message.text) # Extracted textual content

print(
    "Serializable:",
    BaseMessage.is_lc_serializable() # Check LangChain serialization support
)

print(
    "Namespace:",
    BaseMessage.get_lc_namespace() # Get LangChain serialization namespace
)


system_message = SystemMessage(
    content="You are a helpful assistant." # System instruction
)

prompt = system_message + message # Create a ChatPromptTemplate

print("\nCombined Prompt:")
print(prompt)


print("\nPretty Representation:")
print(
    message.pretty_repr(
        html=False # Use plain-text formatting
    )
)


print("\nPretty Print:")
message.pretty_print() # Print the formatted message

Content: Explain SQL joins.
Additional kwargs: {'priority': 'high'}
Response metadata: {'model_name': 'example-model', 'token_count': 10}
Type: human
Name: student
ID: 101
Custom field: custom value
Content blocks: [{'type': 'text', 'text': 'Explain SQL joins.'}]
Text: Explain SQL joins.
Serializable: True
Namespace: ['langchain', 'schema', 'messages']

Combined Prompt:
input_variables=[] input_types={} partial_variables={} messages=[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain SQL joins.', additional_kwargs={'priority': 'high'}, response_metadata={'model_name': 'example-model', 'token_count': 10}, name='student', id='101', custom_field='custom value')]

Pretty Representation:
================================ Human Message =================================
Name: student

Explain SQL joins.

Pretty Print:
================================ Human Message =================================
Name: student

Expl

# BaseMessageChunk: `BaseMessage`
`BaseMessageChunk` represents a partial message produced during streaming. Multiple message chunks can be combined to form a complete message.

## Methods
1. `__add__`:= Combines the current message chunk with another chunk or a list of chunks.

   It merges message content, additional keyword arguments, and response metadata. A `TypeError` is raised when the supplied value is not a message chunk or a list of message chunks.
   
   ```python
   __add__(
       self,
       other: Any  # Message chunk or list of message chunks to combine
   ) -> BaseMessageChunk
   ```

## Functions
1. `merge_content`:= Merges multiple message-content values into one string or list.

   Strings are concatenated directly. Lists are merged, and a string may be appended to the final string element of an existing list.

   ```python
   merge_content(
       first_content: str | list[str | dict[Any, Any]],  # First content value
       *contents: str | list[str | dict[Any, Any]]  # Additional content values
   ) -> str | list[str | dict[Any, Any]]
   ```

2. `message_to_dict`:= Converts one message into a dictionary containing its type and serialized data.
   ```python
   message_to_dict(
       message: BaseMessage  # Message to serialize
   ) -> dict[str, Any]
   ```

3. `messages_to_dict`:= Converts a sequence of messages into a list of serialized dictionaries.
   ```python
   messages_to_dict(
       messages: Sequence[BaseMessage]  # Messages to serialize
   ) -> list[dict[str, Any]]
   ```

4. `get_msg_title_repr`:= Creates a centered title surrounded by equal signs.
   The title can optionally be formatted in bold.
   ```python
   get_msg_title_repr(
       title: str,  # Title to format
       *,
       bold: bool = False  # Whether to make the title bold
   ) -> str
   ```


In [2]:
from pprint import pprint

from langchain_core.messages import (
    AIMessage,
    AIMessageChunk,
    HumanMessage,
    message_to_dict,
    messages_to_dict,
)
from langchain_core.messages.base import (
    BaseMessageChunk,
    get_msg_title_repr,
    merge_content,
)


# Create partial streaming message chunks
chunk1 = AIMessageChunk(
    content="LangChain ",
    additional_kwargs={"source": "model"}, # Provider-specific data
    response_metadata={"model_name": "example-model"} # Response metadata
)

chunk2 = AIMessageChunk(
    content="supports ",
    additional_kwargs={"format": "text"}, # Additional provider data
    response_metadata={"chunk_number": 2} # Metadata for second chunk
)

chunk3 = AIMessageChunk(
    content="streaming.",
    additional_kwargs={"streaming": True}, # Streaming information
    response_metadata={"completed": True} # Completion metadata
)


# Combine one chunk with another chunk
combined_chunk = chunk1 + chunk2

# Combine a chunk with a list of chunks
complete_chunk = combined_chunk + [chunk3]

print(get_msg_title_repr("Combined Message Chunk"))

print("Content:", complete_chunk.content)
print("Additional kwargs:", complete_chunk.additional_kwargs)
print("Response metadata:", complete_chunk.response_metadata)
print(
    "Is BaseMessageChunk:",
    isinstance(complete_chunk, BaseMessageChunk)
)


# Demonstrate invalid chunk addition
try:
    chunk1 + 100 # Invalid value
except TypeError as error:
    print("\nTypeError:", error)


# Merge string content
merged_string = merge_content(
    "Hello, ", # First content value
    "welcome ", # Additional content value
    "to LangChain." # Additional content value
)

print(get_msg_title_repr("Merged String Content"))
print(merged_string)


# Merge list-based and string-based content
merged_list = merge_content(
    ["First block. "], # First content value
    "Second block. ", # Additional string content
    [{"type": "text", "text": "Third block."}] # Additional content block
)

print(get_msg_title_repr("Merged List Content"))
pprint(merged_list)


# Create complete messages
human_message = HumanMessage(
    content="What is streaming?",
    name="student",
    id="message-101"
)

ai_message = AIMessage(
    content=complete_chunk.content,
    id="message-102"
)


# Convert one message into a dictionary
single_message_dict = message_to_dict(
    human_message # Message to serialize
)

print(get_msg_title_repr("Single Serialized Message"))
pprint(single_message_dict)


# Convert multiple messages into dictionaries
multiple_message_dicts = messages_to_dict(
    [human_message, ai_message] # Messages to serialize
)

print(get_msg_title_repr("Multiple Serialized Messages"))
pprint(multiple_message_dicts)


# Create a bold title representation
bold_title = get_msg_title_repr(
    "Program Completed", # Title to format
    bold=True # Format the title in bold
)

print(bold_title)

============================ Combined Message Chunk ============================
Content: LangChain supports streaming.
Additional kwargs: {'source': 'model', 'format': 'text', 'streaming': True}
Response metadata: {'model_name': 'example-model', 'chunk_number': 2, 'completed': True}
Is BaseMessageChunk: True

TypeError: unsupported operand type(s) for +: "AIMessageChunk" and "int"
============================ Merged String Content =============================
Hello, welcome to LangChain.
============================= Merged List Content ==============================
['First block. Second block. ', {'text': 'Third block.', 'type': 'text'}]
========================== Single Serialized Message ===========================
{'data': {'additional_kwargs': {},
          'content': 'What is streaming?',
          'id': 'message-101',
          'name': 'student',
          'response_metadata': {},
          'type': 'human'},
 'type': 'human'}
========================= Multiple Serialized Mess